> **Version corrigée** — ce notebook contient le code complet de tous les exercices, exécuté de bout en bout, ainsi qu'un élément de réponse pour chaque question d'observation. La version étudiant (à compléter soi-même) est téléchargeable depuis la page du cours.


# Procédure expérimentale et mesures de performance

**Notebook 3/9 — Introduction à l'apprentissage supervisé**
*Guillaume Metzler — Université Lyon 2 (L3 MIASHS → Master)*

Un modèle qui obtient une faible erreur sur les données qui ont servi à
l'entraîner ne dit rien, à lui seul, de sa capacité à généraliser. Ce
notebook porte sur la mécanique qui permet d'estimer honnêtement cette
capacité et de choisir entre plusieurs réglages d'un modèle sur cette seule
base :

- le découpage train / validation / test et le réglage d'hyperparamètres
  (`GridSearchCV`),
- la validation croisée à k plis,
- les mesures de performance en régression (MSE, RMSE, MAE) et en
  classification (matrice de confusion, accuracy, précision, rappel,
  F-mesure, courbe ROC, AUC, et quelques mesures moins usuelles adaptées aux
  données déséquilibrées).

Le risque empirique, le risque vrai et le compromis biais-variance en tant
que tels sont traités dans le notebook précédent ; on prend ici pour acquis
qu'un modèle trop complexe peut sur-apprendre, et on se concentre sur
*comment le détecter et le mesurer* correctement.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import (
    make_regression, make_classification, load_wine, load_diabetes,
    load_breast_cancer,
)
from sklearn.model_selection import (
    train_test_split, KFold, cross_val_score, GridSearchCV,
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score,
)

plt.rcParams["figure.figsize"] = (6, 4)
print("Bibliothèques chargées.")



## 1. Entraîner et régler un modèle : train / validation / test

Une fois un modèle appris, la seule question qui compte est sa performance
sur des données qu'il n'a pas vues. On réserve donc, avant tout
entraînement, un **ensemble de test** que le modèle ne touchera qu'une
seule fois, à la toute fin.

Cela ne suffit pas dès que le modèle dépend d'un hyperparamètre à régler
(profondeur d'un arbre, nombre de voisins, régularisation...) : comparer
les valeurs candidates directement sur le test reviendrait à s'en servir
pour choisir le modèle, et l'estimation finale serait trop optimiste. On
réserve donc, à l'intérieur du train, un **ensemble de validation** :

1. séparer les données en `train` (≈ 70%) et `test` (≈ 30%) ;
2. séparer `train` en `learning` (≈ 80% de train) et `validation`
   (≈ 20% de train) ;
3. pour chaque valeur candidate de l'hyperparamètre, apprendre sur
   `learning`, évaluer sur `validation` ;
4. garder la valeur qui donne la meilleure performance de validation ;
5. ré-entraîner un modèle avec cette valeur sur `train` en entier ;
6. évaluer ce modèle final, une seule fois, sur `test`.


In [ ]:

X_syn, y_syn = make_regression(n_samples=300, n_features=5, noise=12.0, random_state=0)

X_train, X_test, y_train, y_test = train_test_split(X_syn, y_syn, test_size=0.3, random_state=0)
X_learn, X_val, y_learn, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

reg = LinearRegression().fit(X_learn, y_learn)

print(f"MSE learning   : {mean_squared_error(y_learn, reg.predict(X_learn)):.1f}  (n={len(y_learn)})")
print(f"MSE validation : {mean_squared_error(y_val, reg.predict(X_val)):.1f}  (n={len(y_val)})")
print(f"MSE test       : {mean_squared_error(y_test, reg.predict(X_test)):.1f}  (n={len(y_test)})")



### Exercice 1 — Découpage train / validation / test sur `load_diabetes`

Faites la même chose sur `load_diabetes()` (régression : score de
progression du diabète) : 70% train / 30% test, puis 80% learning / 20%
validation à l'intérieur du train. Entraînez une `LinearRegression` sur
`learning`, et affichez la MSE sur `learning`, `validation` et `test`.


In [ ]:

X_diab, y_diab = load_diabetes(return_X_y=True)

X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab, test_size=0.3, random_state=0,
)
X_learn_d, X_val_d, y_learn_d, y_val_d = train_test_split(
    X_train_d, y_train_d, test_size=0.2, random_state=0,
)

reg_d = LinearRegression().fit(X_learn_d, y_learn_d)

mse_learn = mean_squared_error(y_learn_d, reg_d.predict(X_learn_d))
mse_val = mean_squared_error(y_val_d, reg_d.predict(X_val_d))
mse_test = mean_squared_error(y_test_d, reg_d.predict(X_test_d))

print(f"MSE learning   : {mse_learn:.1f}   (n={len(y_learn_d)})")
print(f"MSE validation : {mse_val:.1f}   (n={len(y_val_d)})")
print(f"MSE test       : {mse_test:.1f}   (n={len(y_test_d)})")

print(f"\nProportions : train={len(y_train_d)/len(y_diab):.0%}, "
      f"test={len(y_test_d)/len(y_diab):.0%}")
print(f"Dans train  : learning={len(y_learn_d)/len(y_train_d):.0%}, "
      f"validation={len(y_val_d)/len(y_train_d):.0%}")



## 2. Validation croisée

Le découpage learning/validation a un défaut : la validation ne repose que
sur une petite partie des données, potentiellement une faible portion de
la distribution sous-jacente. La **validation croisée à k plis** (k-fold)
corrige ça en faisant tourner le rôle de validation sur k blocs disjoints
du train :

1. on découpe `train` en k blocs (folds) de taille égale ;
2. à chaque tour, k-1 blocs servent à apprendre et le bloc restant sert à
   valider, ce qui donne une performance $p_j$ ;
3. la performance retenue est la moyenne
   $s = \frac{1}{k}\sum_{j=1}^k p_j$ (l'écart-type des $p_j$ donne une idée
   de la stabilité de l'estimation).

C'est plus coûteux — il faut apprendre k modèles au lieu d'un — mais chaque
exemple sert exactement une fois à la validation, contre le tout ou rien
d'un unique découpage.


In [ ]:

X_cls, y_cls = make_classification(n_samples=300, n_features=10, n_informative=5,
                                    random_state=0)
modele = LogisticRegression(max_iter=2000)

scores_splits = []
for rs in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X_cls, y_cls, test_size=0.3, random_state=rs)
    modele_split = clone(modele).fit(X_tr, y_tr)
    scores_splits.append(accuracy_score(y_te, modele_split.predict(X_te)))
scores_splits = np.array(scores_splits)

scores_cv = cross_val_score(modele, X_cls, y_cls, cv=5)

fig, ax = plt.subplots()
ax.boxplot([scores_splits, scores_cv], tick_labels=["20 train/test aléatoires", "5-fold CV"])
ax.set_ylabel("Accuracy")
ax.set_title("Variabilité de l'estimation selon la stratégie de découpage")
plt.show()

print(f"Train/test (20 tirages) : moyenne={scores_splits.mean():.3f}, std={scores_splits.std():.3f}")
print(f"5-fold CV               : moyenne={scores_cv.mean():.3f}, std={scores_cv.std():.3f}")


$$ $$

**Question 1 :** Comparez la dispersion des deux boîtes à moustaches. Combien d'évaluations sur des exemples de test distincts entrent dans le calcul de chaque moyenne (20 tirages indépendants d'un côté, 5 plis de l'autre) ? Qu'est-ce que cela vous dit sur la fiabilité d'une estimation obtenue à partir d'un seul découpage train/test ?

$$ $$

*Éléments de réponse.* Les 20 tirages train/test aléatoires sont nettement plus dispersés que les 5 scores de la validation croisée : chaque tirage ne repose que sur un découpage particulier, dont le résultat dépend fortement du hasard (quels exemples tombent dans le test cette fois-ci). La validation croisée, elle, cumule les évaluations sur l'ensemble des données (chaque exemple sert une fois de test, sur l'un des 5 plis) : sa moyenne est une estimation plus stable, pour un coût de calcul comparable ici (5 entraînements contre 20). Un score obtenu à partir d'un unique découpage train/test doit donc être interprété avec prudence : il peut être optimiste ou pessimiste selon le tirage, sans qu'on le sache sans y regarder à deux fois.

In [ ]:

valeurs_k = [2, 3, 5, 10, 20, 50]
moyennes, ecarts_types = [], []

for k in valeurs_k:
    s = cross_val_score(modele, X_cls, y_cls, cv=k)
    moyennes.append(s.mean())
    ecarts_types.append(s.std())

fig, ax = plt.subplots()
ax.errorbar(valeurs_k, moyennes, yerr=ecarts_types, fmt="o-", capsize=4)
ax.set_xlabel("k (nombre de plis)")
ax.set_ylabel("Accuracy moyenne (barres : écart-type entre plis)")
ax.set_title("Effet de k sur l'estimation de la performance")
plt.show()

for k, m, s in zip(valeurs_k, moyennes, ecarts_types):
    print(f"k={k:>2d} -> moyenne={m:.3f}, std entre plis={s:.3f}, "
          f"taille d'un bloc de validation ~{len(X_cls)//k}")


$$ $$

**Question 2 :** Que devient la taille de chaque bloc de validation quand k augmente ? Et le nombre de modèles à entraîner pour une validation croisée complète ? Comment ces deux effets se traduisent-ils sur le graphique (position de la moyenne, taille des barres d'erreur) ?

$$ $$

*Éléments de réponse.* Quand k augmente, chaque bloc de validation représente une fraction 1/k du train : il devient plus petit, donc chaque score individuel $p_j$ est calculé sur moins d'exemples et est plus sensible au hasard de ce bloc précis (l'écart-type entre plis a tendance à rester notable, voire à augmenter). En contrepartie, l'ensemble d'apprentissage de chaque pli est plus grand (proche du train complet quand k est grand) : le modèle appris à chaque tour se rapproche de celui qu'on obtiendrait avec tout le train, ce qui réduit en général le biais pessimiste de la moyenne. Le coût de calcul, lui, croît linéairement avec k puisqu'il faut apprendre un modèle par pli.


### Régler un hyperparamètre avec `GridSearchCV`

La boucle « pour chaque valeur d'hyperparamètre, faire une validation
croisée » est un cas d'usage si fréquent que scikit-learn le fournit clé en
main : `GridSearchCV` combine l'exploration d'une grille de valeurs et la
validation croisée, retient la meilleure combinaison (`best_params_`,
`best_score_`), et ré-entraîne automatiquement le modèle final sur tout le
jeu fourni (`refit=True` par défaut).


In [ ]:

X_w, y_w = load_wine(return_X_y=True)
X_w = StandardScaler().fit_transform(X_w)

grille = {"n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21]}
recherche = GridSearchCV(KNeighborsClassifier(), grille, cv=5)
recherche.fit(X_w, y_w)

print("Meilleur n_neighbors :", recherche.best_params_["n_neighbors"])
print(f"Meilleur score CV    : {recherche.best_score_:.3f}")

resultats = recherche.cv_results_
plt.figure()
plt.errorbar(grille["n_neighbors"], resultats["mean_test_score"],
             yerr=resultats["std_test_score"], fmt="o-", capsize=4)
plt.xlabel("n_neighbors")
plt.ylabel("Accuracy moyenne (CV)")
plt.title("GridSearchCV : score moyen par valeur de l'hyperparamètre")
plt.show()



### Exercice 2 — `GridSearchCV`

Sur `load_breast_cancer`, séparez d'abord un jeu de test (30%,
`stratify=y`) que vous ne toucherez qu'à la fin. Sur le reste (`train`),
utilisez `GridSearchCV` pour régler `max_depth` d'un
`DecisionTreeClassifier` (grille `[1, 2, 3, 5, 8, 12, None]`, `cv=5`).
Affichez le meilleur `max_depth` et le meilleur score de validation
croisée, puis évaluez le modèle final (`recherche.best_estimator_`) sur le
jeu de test mis de côté.


In [ ]:

X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    X_bc, y_bc, test_size=0.3, random_state=0, stratify=y_bc,
)

grille_bc = {"max_depth": [1, 2, 3, 5, 8, 12, None]}
recherche_bc = GridSearchCV(DecisionTreeClassifier(random_state=0), grille_bc, cv=5)
recherche_bc.fit(X_train_bc, y_train_bc)

print("Meilleur max_depth :", recherche_bc.best_params_["max_depth"])
print(f"Meilleur score CV  : {recherche_bc.best_score_:.3f}")

meilleur_modele = recherche_bc.best_estimator_
acc_test = accuracy_score(y_test_bc, meilleur_modele.predict(X_test_bc))
print(f"Accuracy sur le test (jamais utilisé pendant le réglage) : {acc_test:.3f}")



## 3. Mesures de performance en régression

Pour $m$ exemples, avec $y_i$ la vraie valeur et $\hat y_i$ la valeur
prédite :

$$
\text{MSE} = \frac{1}{m}\sum_{i=1}^m (y_i-\hat y_i)^2, \qquad
\text{RMSE} = \sqrt{\text{MSE}}, \qquad
\text{MAE} = \frac{1}{m}\sum_{i=1}^m |y_i-\hat y_i|.
$$

La MSE est convexe et différentiable, donc pratique à optimiser
directement — mais elle élève l'erreur au carré : une seule observation
très mal prédite pèse énormément dans la moyenne. La MAE, fondée sur la
norme $\ell_1$, est plus robuste aux valeurs aberrantes, au prix d'être
moins agréable à optimiser (non différentiable en 0). D'autres critères
existent pour comparer des modèles de régression — les critères AIC, BIC
et le $R^2$ — mais ne sont pas développés ici.


In [ ]:

rng = np.random.RandomState(1)
y_vrai = rng.normal(loc=50, scale=5, size=100)
y_pred = y_vrai + rng.normal(scale=2, size=100)

y_pred_outlier = y_pred.copy()
y_pred_outlier[0] += 60  # une seule prediction tres mauvaise

for nom, yp in [("Sans outlier", y_pred), ("Avec un outlier", y_pred_outlier)]:
    mse = mean_squared_error(y_vrai, yp)
    mae = mean_absolute_error(y_vrai, yp)
    print(f"{nom:18s} -> MSE={mse:7.2f}  RMSE={np.sqrt(mse):6.2f}  MAE={mae:6.2f}")



Une seule prédiction décalée de 60 fait exploser la MSE (et donc la RMSE),
alors que la MAE n'est que faiblement affectée : c'est la sensibilité aux
valeurs aberrantes évoquée en cours.

### Exercice 3 — Sensibilité aux outliers sur `load_diabetes`

Entraînez une `LinearRegression` sur `load_diabetes` (train/test 70/30).
Calculez MSE, RMSE, MAE « normales » sur le test. Créez ensuite une copie
des prédictions dans laquelle vous ajoutez 200 aux 3 premières valeurs
prédites, recalculez les trois mesures sur cette version modifiée, et
affichez, pour chacune, le ratio (avec outliers / sans outliers).


In [ ]:

X_diab2, y_diab2 = load_diabetes(return_X_y=True)
X_train_d2, X_test_d2, y_train_d2, y_test_d2 = train_test_split(
    X_diab2, y_diab2, test_size=0.3, random_state=0,
)

reg_d2 = LinearRegression().fit(X_train_d2, y_train_d2)
y_pred_d2 = reg_d2.predict(X_test_d2)

mse_normal = mean_squared_error(y_test_d2, y_pred_d2)
rmse_normal = np.sqrt(mse_normal)
mae_normal = mean_absolute_error(y_test_d2, y_pred_d2)

y_pred_outlier2 = y_pred_d2.copy()
y_pred_outlier2[:3] += 200

mse_out = mean_squared_error(y_test_d2, y_pred_outlier2)
rmse_out = np.sqrt(mse_out)
mae_out = mean_absolute_error(y_test_d2, y_pred_outlier2)

print(f"Sans outlier  -> MSE={mse_normal:.1f}  RMSE={rmse_normal:.1f}  MAE={mae_normal:.1f}")
print(f"Avec outliers -> MSE={mse_out:.1f}  RMSE={rmse_out:.1f}  MAE={mae_out:.1f}")

ratio_mse = mse_out / mse_normal
ratio_rmse = rmse_out / rmse_normal
ratio_mae = mae_out / mae_normal
print(f"\nRatio (avec/sans) : MSE x{ratio_mse:.2f}, RMSE x{ratio_rmse:.2f}, MAE x{ratio_mae:.2f}")

assert ratio_mse >= ratio_mae, "La MSE devrait etre plus sensible que la MAE aux outliers"
print("La MSE (et la RMSE) est nettement plus sensible aux outliers que la MAE.")



## 4. Mesures de performance en classification

### 4.1 Matrice de confusion et mesures dérivées

Pour un problème binaire (classe positive $y=1$), la matrice de confusion
croise la vraie classe et la classe prédite :

|                     | Prédit positif | Prédit négatif |
|---------------------|:--------------:|:--------------:|
| **Réel positif**    | TP             | FN             |
| **Réel négatif**    | FP             | TN             |

$$
\text{Accuracy} = \frac{TP+TN}{m}, \qquad
\text{Précision} = \frac{TP}{TP+FP}, \qquad
\text{Rappel (Sensibilité)} = \frac{TP}{TP+FN}, \qquad
F_1 = \frac{2\,\text{Précision}\times\text{Rappel}}{\text{Précision}+\text{Rappel}}.
$$

La précision répond à « parmi les exemples prédits positifs, combien le
sont vraiment ? », le rappel à « parmi les exemples réellement positifs,
combien le modèle en retrouve-t-il ? ». L'accuracy, elle, mélange les deux
classes sans distinction, ce qui devient trompeur dès que les classes sont
déséquilibrées : sur 100 000 exemples dont 1 000 positifs, un modèle qui
prédit systématiquement « négatif » atteint déjà 99% d'accuracy sans
jamais détecter le moindre positif.


In [ ]:

X_w3, y_w3 = make_classification(
    n_samples=400, n_features=8, n_informative=4, n_clusters_per_class=1,
    class_sep=0.8, flip_y=0.1, random_state=0,
)
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_w3, y_w3, test_size=0.3, random_state=0, stratify=y_w3,
)

pipe3 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr3, y_tr3)
y_pred3 = pipe3.predict(X_te3)

cm = confusion_matrix(y_te3, y_pred3)
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["classe 0", "classe 1"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matrice de confusion - jeu simulé équilibré")
plt.show()

print(f"Accuracy  : {accuracy_score(y_te3, y_pred3):.3f}")
print(f"Précision : {precision_score(y_te3, y_pred3):.3f}")
print(f"Rappel    : {recall_score(y_te3, y_pred3):.3f}")
print(f"F1        : {f1_score(y_te3, y_pred3):.3f}")


In [ ]:

X_imb, y_imb = make_classification(
    n_samples=2000, n_features=10, n_informative=6, n_clusters_per_class=1,
    class_sep=1.0, weights=[0.97, 0.03], flip_y=0.0, random_state=0,
)
print(f"Proportion de positifs : {y_imb.mean():.1%}")

X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=0, stratify=y_imb,
)
pred_trivial = np.zeros_like(y_te_i)  # predit toujours la classe majoritaire (0)

modele_reel = LogisticRegression(max_iter=2000).fit(X_tr_i, y_tr_i)
pred_reel = modele_reel.predict(X_te_i)

for nom, pred in [("Modèle trivial (toujours 0)", pred_trivial),
                   ("LogisticRegression", pred_reel)]:
    acc = accuracy_score(y_te_i, pred)
    rec = recall_score(y_te_i, pred, zero_division=0)
    f1 = f1_score(y_te_i, pred, zero_division=0)
    print(f"{nom:28s} -> accuracy={acc:.3f}  rappel={rec:.3f}  F1={f1:.3f}")


$$ $$

**Question 3 :** Le modèle trivial obtient-il une bonne accuracy ? Et un bon F1 ? Qu'est-ce que le F1 capture ici que l'accuracy ne capture pas ?

$$ $$

*Éléments de réponse.* Le modèle trivial atteint une accuracy très élevée, proche de la proportion de négatifs (donc plus de 95% ici), simplement parce que la classe positive est rare : il n'a rien besoin d'apprendre pour ça. Son rappel et son F1 sont en revanche nuls, puisqu'il ne détecte jamais aucun positif — ce que l'accuracy, moyenne globale des bonnes prédictions sur les deux classes confondues, ne reflète pas du tout. Le F1, moyenne harmonique de précision et rappel, s'effondre dès que l'une des deux composantes est nulle : c'est un signal beaucoup plus fiable que l'accuracy en contexte déséquilibré.

In [ ]:

y_true_jouet = np.array([1, 1, 1, 0, 0, 0, 1, 0, 1, 0])
y_pred_jouet = np.array([1, 0, 1, 0, 1, 0, 1, 0, 0, 0])

TP = int(np.sum((y_true_jouet == 1) & (y_pred_jouet == 1)))
FP = int(np.sum((y_true_jouet == 0) & (y_pred_jouet == 1)))
FN = int(np.sum((y_true_jouet == 1) & (y_pred_jouet == 0)))

precision_manuelle = TP / (TP + FP)
rappel_manuel = TP / (TP + FN)
f1_manuel = 2 * precision_manuelle * rappel_manuel / (precision_manuelle + rappel_manuel)

print(f"À la main    : precision={precision_manuelle:.3f}, rappel={rappel_manuel:.3f}, f1={f1_manuel:.3f}")
print(f"scikit-learn : precision={precision_score(y_true_jouet, y_pred_jouet):.3f}, "
      f"rappel={recall_score(y_true_jouet, y_pred_jouet):.3f}, "
      f"f1={f1_score(y_true_jouet, y_pred_jouet):.3f}")

assert abs(precision_manuelle - precision_score(y_true_jouet, y_pred_jouet)) < 1e-9
assert abs(rappel_manuel - recall_score(y_true_jouet, y_pred_jouet)) < 1e-9
assert abs(f1_manuel - f1_score(y_true_jouet, y_pred_jouet)) < 1e-9
print("Vérification OK.")



### Exercice 4 — Précision / rappel / F1 « à la main », sur de vraies prédictions

Écrivez une fonction `precision_rappel_f1(y_true, y_pred)` qui reproduit le
calcul ci-dessus (traitez le cas de division par zéro en renvoyant 0.0).
Entraînez une `LogisticRegression` sur `load_breast_cancer` (train/test
70/30, `StandardScaler`), appliquez votre fonction aux prédictions du test,
et vérifiez le résultat avec `precision_score`, `recall_score`, `f1_score`
de scikit-learn (`assert`).


In [ ]:

def precision_rappel_f1(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    rappel = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = (2 * precision * rappel / (precision + rappel)
          if (precision + rappel) > 0 else 0.0)
    return precision, rappel, f1


X_bc4, y_bc4 = load_breast_cancer(return_X_y=True)
X_tr4, X_te4, y_tr4, y_te4 = train_test_split(
    X_bc4, y_bc4, test_size=0.3, random_state=0, stratify=y_bc4,
)
pipe4 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr4, y_tr4)
y_pred4 = pipe4.predict(X_te4)

p, r, f = precision_rappel_f1(y_te4, y_pred4)
print(f"Ma fonction  : precision={p:.3f}, rappel={r:.3f}, f1={f:.3f}")
print(f"scikit-learn : precision={precision_score(y_te4, y_pred4):.3f}, "
      f"rappel={recall_score(y_te4, y_pred4):.3f}, f1={f1_score(y_te4, y_pred4):.3f}")

assert abs(p - precision_score(y_te4, y_pred4)) < 1e-9
assert abs(r - recall_score(y_te4, y_pred4)) < 1e-9
assert abs(f - f1_score(y_te4, y_pred4)) < 1e-9
print("\nVérification OK : les deux implémentations coïncident.")



### 4.2 Courbe ROC et AUC

Quand le modèle renvoie un score (une probabilité) plutôt qu'une décision
brute, on peut faire varier le seuil de décision au lieu de le fixer à
0.5 : un seuil plus bas classe davantage d'exemples en positif (rappel
plus élevé, précision plus basse en général), un seuil plus haut fait
l'inverse. La **courbe ROC** trace, pour tous les seuils possibles, le
taux de vrais positifs $\text{TPR} = \text{Rappel}$ en fonction du taux de
faux positifs $\text{FPR} = \frac{FP}{FP+TN}$. L'**AUC** (aire sous cette
courbe) résume la courbe en une valeur entre 0.5 (aléatoire) et 1
(classifieur parfait) : elle mesure la qualité du classement produit par
les scores, indépendamment du choix d'un seuil particulier.


In [ ]:

X_w5, y_w5 = make_classification(
    n_samples=400, n_features=8, n_informative=4, n_clusters_per_class=1,
    class_sep=0.9, flip_y=0.08, random_state=3,
)
X_tr5, X_te5, y_tr5, y_te5 = train_test_split(
    X_w5, y_w5, test_size=0.3, random_state=0, stratify=y_w5,
)
pipe5 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr5, y_tr5)
scores5 = pipe5.predict_proba(X_te5)[:, 1]

fpr, tpr, seuils = roc_curve(y_te5, scores5)
auc5 = roc_auc_score(y_te5, scores5)

plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, label=f"LogisticRegression (AUC={auc5:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Modèle aléatoire")

for seuil_cible in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pred_seuil = (scores5 >= seuil_cible).astype(int)
    fpr_pt = np.sum((y_te5 == 0) & (pred_seuil == 1)) / np.sum(y_te5 == 0)
    tpr_pt = np.sum((y_te5 == 1) & (pred_seuil == 1)) / np.sum(y_te5 == 1)
    plt.scatter(fpr_pt, tpr_pt, s=60, zorder=5)
    plt.annotate(f"{seuil_cible}", (fpr_pt, tpr_pt), textcoords="offset points", xytext=(5, 5))

plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR / Rappel)")
plt.title("Courbe ROC : position du point selon le seuil de décision")
plt.legend()
plt.show()


$$ $$

**Question 4 :** En vous déplaçant du seuil 0.9 vers le seuil 0.1, comment évoluent le FPR et le TPR ? Quel seuil choisiriez-vous si l'objectif était de ne rater aucun exemple positif, quitte à multiplier les fausses alertes ?

$$ $$

*Éléments de réponse.* En diminuant le seuil, on classe positif de plus en plus d'exemples : le rappel (TPR) augmente puisqu'on manque moins de vrais positifs, mais le FPR augmente aussi car davantage de négatifs sont classés à tort positifs — on se déplace le long de la courbe vers le coin supérieur droit. Pour ne rater aucun positif, on choisirait un seuil bas (proche de 0.1, voire plus bas), en acceptant le coût en fausses alertes que cela implique. Le compromis à retenir dépend du coût respectif d'un faux négatif et d'un faux positif dans l'application visée : un seuil bas convient à un dépistage médical, un seuil haut à un filtre anti-spam où l'on préfère laisser passer un spam plutôt que bloquer un message légitime.


### Exercice 5 — Courbe ROC et AUC sur un jeu très déséquilibré

Générez un jeu `make_classification` (2000 exemples, `weights=[0.9, 0.1]`,
`n_informative=4`, `random_state=0`), séparez en train/test (70%/30%,
`stratify=y`), entraînez un pipeline `StandardScaler` + `LogisticRegression`.
Récupérez les scores de la classe positive (`predict_proba`), tracez la
courbe ROC avec la diagonale du modèle aléatoire, et affichez l'AUC dans le
titre du graphique.


In [ ]:

X_imb2, y_imb2 = make_classification(
    n_samples=2000, n_features=10, n_informative=4, n_redundant=2,
    weights=[0.9, 0.1], flip_y=0.02, random_state=0,
)
X_tr6, X_te6, y_tr6, y_te6 = train_test_split(
    X_imb2, y_imb2, test_size=0.3, random_state=0, stratify=y_imb2,
)

pipe6 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pipe6.fit(X_tr6, y_tr6)
scores6 = pipe6.predict_proba(X_te6)[:, 1]

fpr6, tpr6, seuils6 = roc_curve(y_te6, scores6)
auc6 = roc_auc_score(y_te6, scores6)

plt.figure(figsize=(5.5, 5))
plt.plot(fpr6, tpr6, label="LogisticRegression")
plt.plot([0, 1], [0, 1], "k--", label="Modèle aléatoire")
plt.xlabel("Taux de faux positifs (FPR)")
plt.ylabel("Taux de vrais positifs (TPR / Rappel)")
plt.title(f"Courbe ROC (AUC = {auc6:.3f})")
plt.legend()
plt.show()

print(f"Proportion de positifs dans le test : {y_te6.mean():.1%}")
print(f"AUC : {auc6:.3f}")
print(f"Accuracy (seuil 0.5) : {accuracy_score(y_te6, pipe6.predict(X_te6)):.3f}")
assert 0.5 <= auc6 <= 1.0



### 4.3 Autres mesures en contexte déséquilibré

Le F1 pondère également précision et rappel. Le F-mesure généralisé
$F_\beta$ permet de déséquilibrer ce compromis :

$$
F_\beta = \frac{(1+\beta^2)\,\text{Précision}\times\text{Rappel}}{\beta^2\,\text{Précision}+\text{Rappel}},
\qquad F_1 \text{ correspond à } \beta=1.
$$

Plus $\beta$ est grand, plus on privilégie le rappel ; plus il est petit,
plus on privilégie la précision. D'autres mesures combinent différemment
sensibilité (rappel) et spécificité
$\big(\text{Spécificité} = \frac{TN}{TN+FP}\big)$ :

$$
\text{CWA}_\alpha = \alpha\times\text{Sensibilité} + (1-\alpha)\times\text{Spécificité},
\qquad
\text{G-mean} = \sqrt{\text{Sensibilité}\times\text{Spécificité}}.
$$

Contrairement à l'accuracy classique, la *Class Weighted Accuracy* (CWA) et
le G-mean donnent le même poids à chaque classe, quel que soit son effectif.


In [ ]:

def mesures_depuis_confusion(TP, FN, FP, TN, beta=1.0, alpha=0.5):
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    rappel = TP / (TP + FN) if (TP + FN) else 0.0
    specificite = TN / (TN + FP) if (TN + FP) else 0.0
    f_beta = ((1 + beta**2) * precision * rappel / (beta**2 * precision + rappel)
              if (precision + rappel) else 0.0)
    cwa = alpha * rappel + (1 - alpha) * specificite
    g_mean = np.sqrt(rappel * specificite)
    return precision, rappel, specificite, f_beta, cwa, g_mean

# Deux classifieurs sur un jeu a 1% de positifs (10 positifs, 990 negatifs)
h1 = dict(TP=3, FN=7, FP=0, TN=990)
h2 = dict(TP=9, FN=1, FP=6, TN=984)

for nom, h in [("h1", h1), ("h2", h2)]:
    p, r, spe, f1, cwa, gmean = mesures_depuis_confusion(**h)
    print(f"{nom} -> precision={p:.2f}  rappel={r:.2f}  specificite={spe:.3f}  "
          f"F1={f1:.2f}  CWA(0.5)={cwa:.3f}  G-mean={gmean:.3f}")



Malgré une précision parfaite, h1 rate 7 des 10 positifs (rappel faible) :
son F1 comme son G-mean restent inférieurs à ceux de h2, qui accepte
quelques faux positifs mais en détecte beaucoup plus.

### Exercice 6 — F2, CWA et G-mean sur un vrai classifieur

Entraînez une `LogisticRegression` sur un jeu `make_classification`
déséquilibré (`weights=[0.95, 0.05]`, `random_state=1`, train/test 70/30,
`stratify=y`). Comparez, sur le test, les prédictions obtenues avec un
seuil de 0.5 et avec un seuil de 0.3 : pour chacun, calculez précision,
rappel, F1, F2 (`beta=2`), CWA (`alpha=0.5`) et G-mean à partir de la
matrice de confusion (`confusion_matrix`). Lequel des deux seuils favorise
le mieux la détection de la classe minoritaire, selon le G-mean et le F2 ?


In [ ]:

def mesures_depuis_confusion(TP, FN, FP, TN, beta=1.0, alpha=0.5):
    precision = TP / (TP + FP) if (TP + FP) else 0.0
    rappel = TP / (TP + FN) if (TP + FN) else 0.0
    specificite = TN / (TN + FP) if (TN + FP) else 0.0
    f_beta = ((1 + beta**2) * precision * rappel / (beta**2 * precision + rappel)
              if (precision + rappel) else 0.0)
    cwa = alpha * rappel + (1 - alpha) * specificite
    g_mean = np.sqrt(rappel * specificite)
    return precision, rappel, specificite, f_beta, cwa, g_mean


X_imb3, y_imb3 = make_classification(
    n_samples=2000, n_features=10, n_informative=4,
    weights=[0.95, 0.05], random_state=1,
)
X_tr7, X_te7, y_tr7, y_te7 = train_test_split(
    X_imb3, y_imb3, test_size=0.3, random_state=1, stratify=y_imb3,
)
pipe7 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr7, y_tr7)
scores7 = pipe7.predict_proba(X_te7)[:, 1]

resultats_seuils = {}
for seuil in [0.5, 0.3]:
    pred = (scores7 >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_te7, pred).ravel()
    p, r, spe, f2, cwa, gmean = mesures_depuis_confusion(tp, fn, fp, tn, beta=2.0, alpha=0.5)
    resultats_seuils[seuil] = dict(precision=p, rappel=r, f2=f2, cwa=cwa, gmean=gmean)
    f1 = mesures_depuis_confusion(tp, fn, fp, tn, beta=1.0)[3]
    print(f"seuil={seuil} -> precision={p:.3f}  rappel={r:.3f}  F1={f1:.3f}  "
          f"F2={f2:.3f}  CWA={cwa:.3f}  G-mean={gmean:.3f}")

meilleur_seuil = max(resultats_seuils, key=lambda s: resultats_seuils[s]["gmean"])
print(f"\nSeuil favorisant le mieux la classe minoritaire (G-mean) : {meilleur_seuil}")



## 5. Retour sur la validation croisée : l'implémenter soi-même

`cross_val_score` cache une mécanique simple : `KFold(n_splits=k,
...).split(X)` renvoie, à chaque tour, les indices d'apprentissage et de
validation du pli courant.


In [ ]:

X_jouet = np.arange(12).reshape(-1, 1)
kf_jouet = KFold(n_splits=4, shuffle=True, random_state=0)

for i, (idx_train, idx_val) in enumerate(kf_jouet.split(X_jouet), start=1):
    print(f"Pli {i} -> train={idx_train.tolist()}, validation={idx_val.tolist()}")



À chaque tour, un bloc différent sert de validation : c'est exactement ce
qu'il faut reproduire, avec un vrai modèle, pour ré-implémenter la
validation croisée à la main.

### Exercice 7 (Master) — Validation croisée « à la main »

Implémentez `k_fold_cv_manuel(estimateur, X, y, k=5, random_state=0)`, qui
utilise `KFold(n_splits=k, shuffle=True, random_state=random_state)`,
entraîne à chaque tour un clone (`sklearn.base.clone`) de `estimateur` sur
les indices d'apprentissage, prédit sur les indices de validation, et
renvoie le tableau des k scores d'accuracy. Comparez, sur `load_wine`
(variables standardisées) avec une `LogisticRegression`, vos scores à ceux
de `cross_val_score` utilisant le **même** découpage (même objet `KFold`
passé en `cv=`), et vérifiez l'égalité avec `np.testing.assert_allclose`.


In [ ]:

def k_fold_cv_manuel(estimateur, X, y, k=5, random_state=0):
    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    scores = []
    for idx_train, idx_val in kf.split(X):
        X_tr, X_val = X[idx_train], X[idx_val]
        y_tr, y_val = y[idx_train], y[idx_val]

        modele_clone = clone(estimateur)
        modele_clone.fit(X_tr, y_tr)

        y_pred = modele_clone.predict(X_val)
        scores.append(accuracy_score(y_val, y_pred))
    return np.array(scores)


X_wine2, y_wine2 = load_wine(return_X_y=True)
X_wine2 = StandardScaler().fit_transform(X_wine2)
estimateur = LogisticRegression(max_iter=2000)

cv_identique = KFold(n_splits=5, shuffle=True, random_state=0)

mes_scores = k_fold_cv_manuel(estimateur, X_wine2, y_wine2, k=5, random_state=0)
scores_sklearn = cross_val_score(estimateur, X_wine2, y_wine2, cv=cv_identique)

print("Mes scores     :", np.round(mes_scores, 3))
print("Scores sklearn :", np.round(scores_sklearn, 3))
print(f"Ma moyenne +/- std      : {mes_scores.mean():.3f} +/- {mes_scores.std():.3f}")
print(f"Moyenne sklearn +/- std : {scores_sklearn.mean():.3f} +/- {scores_sklearn.std():.3f}")

np.testing.assert_allclose(mes_scores, scores_sklearn, atol=1e-8)
print("\nVérification OK : ma validation croisée manuelle reproduit exactement "
      "les scores de cross_val_score.")



## Pour la suite

Ces outils — découpage train/validation/test, validation croisée, mesures
adaptées à la tâche — seront réutilisés tels quels dans tous les notebooks
suivants pour entraîner et comparer k-NN, SVM, arbres de décision, méthodes
ensemblistes... Bien évaluer un modèle est un préalable à toute comparaison
honnête entre plusieurs algorithmes.
